<a href="https://colab.research.google.com/github/noorxwaleed-oss/online-AI-host/blob/main/content_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade --quiet \
    langchain-community \
    langchain-openai \
    chromadb \
    pypdf \
    pandas \
    streamlit \
    python-dotenv

In [ ]:
import os
import json
import uuid
from openai import OpenAI
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import bs4

In [ ]:
# =============================
# 1. Setup
# =============================
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get("OPENROUTER_API_KEY" ),
)

In [ ]:
def load_data(source):
    if source.startswith("http" ):
        print(f"🌐 جاري تحميل الرابط...")
        loader = WebBaseLoader(
            web_path=(source,),
            bs_kwargs=dict(parse_only=bs4.SoupStrainer(name=("article", "h1", "h2", "h3", "p")))
        )
    else:
        print(f"📄 جاري تحميل ملف PDF...")
        loader = PyPDFLoader(source)
    return loader.load()


In [ ]:
import hashlib
import os


def get_embedding_func():
    return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def create_vectorstore(chunks, embedding_func, vectorstore_path):
    print(f"🛠️ جاري إنشاء قاعدة بيانات في المجلد: {vectorstore_path}")


    ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, doc.page_content)) for doc in chunks]
    unique_ids, unique_chunks = set(), []
    for chunk, id in zip(chunks, ids):
        if id not in unique_ids:
            unique_ids.add(id)
            unique_chunks.append(chunk)

    vectorstore = Chroma.from_documents(
        documents=unique_chunks,
        ids=list(unique_ids),
        embedding=embedding_func,
        persist_directory=vectorstore_path
    )

    return vectorstore


In [ ]:
source_input = ""

file_name = os.path.basename(source_input)
folder_suffix = hashlib.md5(file_name.encode()).hexdigest()[:8]
unique_path = f"vectorstore_{folder_suffix}"

embedding_func = get_embedding_func()
pages = load_data(source_input)


if not os.path.exists(unique_path):
    vectorstore = create_vectorstore(pages, embedding_func, unique_path)
else:
    print(f"♻️ تم العثور على ذاكرة سابقة لهذا الملف في: {unique_path}")
    vectorstore = Chroma(persist_directory=unique_path, embedding_function=embedding_func)


In [ ]:


def run_smart_analysis(vectorstore):
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 15}
    )

    relevant_chunks = retriever.invoke(
        "podcast main topics discussions arguments insights stories"
    )

    context = "\n\n".join(
        [chunk.page_content for chunk in relevant_chunks]
    )

    ANALYZER_PROMPT =  f"""
    You are a Content Analyzer Agent. Your goal is to prepare material for a professional podcast.

    YOUR TASK:
    - Detect and preserve the ORIGINAL language of the context.
    - Analyze the FULL context deeply.
    - Extract 20 to 30 unique, high-value topics from the context.

    INSTRUCTION FOR SHORT CONTENT:
    - Assess the depth of the provided 'Context'.
    - Focus only on information explicitly or strongly implied in context.
    - If context is rich:
      extract 20-30 topics
      set "content_status" to "sufficient"

    - If context is limited:
      extract only 10 strong topics
      set "content_status" to "limited"

    Rules:
    - NEVER translate the content.
    - ALL output MUST remain in the SAME language as the context.
    - Avoid hallucination.
    - Do not invent information.
    - The entire output content MUST be in language of the provided 'contex'.
    - STRICT PROHIBITION: Do NOT use any Chinese characters (中文), even if the model defaults to it.
    - Focus on facts, stories, and controversial points suitable for a professional dialogue.
    - Return ONLY valid JSON.




    Context: {context}

    Format:
    {{
      "source_language": "",
      "content_status": "",
      "topics": [
        {{
          "title": "Topic Title",
          "key_points": ["Point 1", "Point 2"],
          "insight": "Deep takeaway",
          "discussion_angles": ["Question for the host", "Counter-argument"]
        }}
      ]
    }}
    """

    completion = client.chat.completions.create(
         #model="qwen/qwen-2.5-72b-instruct",
         model ="meta-llama/llama-3.3-70b-instruct",
         messages=[{"role": "user", "content": ANALYZER_PROMPT}],temperature=0.3


    )
    return completion.choices[0].message.content

In [ ]:
#another prompt for translation cause llm doesnt give best answer when doing both analyzing and tranlating at same time 😓


user_language_choice = ""
def translator(analysis_json, target_lang=user_language_choice):

    TRANSLATOR_PROMPT = f"""
    You are a professional JSON translator.

    TASK:
    Translate the JSON content into {target_lang}.

    - Preserve ONLY:
    1. Proper nouns (names of people, places, organizations)
    2. Brand names
    3. Fixed international terms that do not have a common translation in {target_lang}

    STRICT RULES:
    - The goal is natural, human-like bilingual fluency, not literal translation.
    - Preserve JSON structure EXACTLY.
    - Do NOT summarize.
    - Do NOT add explanations.
    - Do NOT remove fields.
    - Translate ONLY text values.
    - Keep tone natural and professional.
    - Return ONLY valid JSON.

    JSON:
    {json.dumps(analysis_json, ensure_ascii=False)}
    """

    completion = client.chat.completions.create(
        model="meta-llama/llama-3.3-70b-instruct",
        messages=[
            {
                "role": "user",
                "content": TRANSLATOR_PROMPT
            }
        ],
        temperature=0.1
    )

    return completion.choices[0].message.content


In [ ]:
import json
import re

try:
    print("🧠 جاري تحليل المحتوى الآن... قد يستغرق ذلك بضع ثوانٍ.")

    # 1. Run analyzer
    analysis = run_smart_analysis(vectorstore)

    # 2. Check empty response
    if not analysis or analysis.strip() == "":
        raise ValueError("النموذج لم يُرجع أي محتوى.")

    # 3. Clean markdown once
    analysis = re.sub(r"```json|```", "", analysis).strip()

    # 4. Extract JSON safely
    json_start = analysis.find("{")
    if json_start != -1:
        analysis = analysis[json_start:]

    # 5. Convert to JSON (ONLY ONCE)
    analysis_json = json.loads(analysis)

    # 6. Detect source language safely (fallback added)
    source_lang = analysis_json.get("source_language", "unknown")

    print(f"🌍 لغة المحتوى الأصلية: {source_lang}")

    # 7. Translation decision
    if source_lang.lower() != user_language_choice.lower():

        print(f"🔄 جاري ترجمة التحليل إلى {user_language_choice}...")

        translated_analysis = translator(
            analysis_json,
            target_lang=user_language_choice
        )

        # Clean translation output
        translated_analysis = re.sub(r"```json|```", "", translated_analysis).strip()

        # Extract JSON safely again
        json_start = translated_analysis.find("{")
        if json_start != -1:
            translated_analysis = translated_analysis[json_start:]

        final_analysis = json.loads(translated_analysis)

    else:
        print("✅ لا حاجة للترجمة، لغة المحتوى مطابقة للغة المطلوبة.")
        final_analysis = analysis_json

    # 8. Final output
    print(f"✅ تم التحليل بنجاح! تم استخراج {len(final_analysis['topics'])} موضوعاً.")

    print("-" * 40)
    print(json.dumps(final_analysis, indent=2, ensure_ascii=False))
    print("-" * 40)

except json.JSONDecodeError as e:
    print("❌ فشل تحويل الناتج إلى JSON صالح.")
    print(f"تفاصيل الخطأ: {e}")

except Exception as e:
    print(f"❌ حدث خطأ أثناء التحليل: {e}")